<a href="https://colab.research.google.com/github/KostantinDubnitskyi/Methodd/blob/main/%D0%9B%D0%A0_9_(%D0%94%D1%83%D0%B1%D0%BD%D0%B8%D1%86%D1%8C%D0%BA%D0%B8%D0%B9_4_8).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Дубницький Костянтин Віталійович  
**Група:** 8
**Дата:** 15.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [ ]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [18]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [19]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [20]:
# TODO 2.1 — Дослідіть набір даних:
# - кількість унікальних користувачів;
# - кількість унікальних фільмів;
# - мінімальне, максимальне, середнє значення rating;
# - пропущені значення (за наявності).

# Ваш код:
n_users = ratings_df['user_id'].nunique()
n_items = ratings_df['item_id'].nunique()

print('Унікальних користувачів:', n_users)
print('Унікальних фільмів:', n_items)
print('Rating: min =', ratings_df['rating'].min(),
      '| max =', ratings_df['rating'].max(),
      '| mean =', round(ratings_df['rating'].mean(), 3))
print('Пропуски:')
print(ratings_df.isna().sum())
print('Розподіл оцінок:')
print(ratings_df['rating'].value_counts().sort_index())

Унікальних користувачів: 943
Унікальних фільмів: 1682
Rating: min = 1.0 | max = 5.0 | mean = 3.53
Пропуски:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64
Розподіл оцінок:
rating
1.0     6110
2.0    11370
3.0    27145
4.0    34174
5.0    21201
Name: count, dtype: int64


**Короткі спостереження щодо даних:**

_Запишіть 2–3 речення про кількість користувачів, фільмів та характер оцінок._

Перевірив набір ml-100k і з'ясував, що в ньому 100 000 оцінок від 943 користувачів на 1682 фільми, пропущених значень немає. Оцінки йдуть від 1 до 5, а в середньому виходить 3.53. Найчастіше ставлять 4, а найрідше ставлять 1.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [21]:
# TODO 3.1 — Створіть train_df та test_df із ratings_df.
# Використайте sklearn_train_test_split(..., test_size=0.2, random_state=42).

# Ваш код:
train_df, test_df = sklearn_train_test_split(ratings_df, test_size=0.2, random_state=42)
print('Train:', train_df.shape, '| Test:', test_df.shape)

Train: (80000, 4) | Test: (20000, 4)


In [22]:
# TODO 3.2 — Підготуйте об’єкти для Surprise:
# 1. reader = Reader(rating_scale=...)
#    Підказка: data.reader.rating_scale
# 2. train_data = Dataset.load_from_df(...)
#    Візьміть тільки user_id, item_id, rating із train_df.
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
# 4. Перевірте розміри train/test.

# Ваш код:
reader = Reader(rating_scale=data.reader.rating_scale)
train_data = Dataset.load_from_df(train_df[['user_id', 'item_id', 'rating']], reader)
testset = list(zip(test_df['user_id'], test_df['item_id'], test_df['rating']))

print('Шкала оцінок:', reader.rating_scale)
print('Train:', len(train_df), '| Test:', len(testset))

Шкала оцінок: (1, 5)
Train: 80000 | Test: 20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [23]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

_Поясніть, які параметри впливають на якість моделей і чому обрали саме ці значення._

Для SVD я розглянув три параметри: n_factors (кількість прихованих факторів, що використовує модель), lr_all (розмір кроків, якими вона вчиться) і reg_all (регуляризація, щоб уникнути перенавчання). Для KNNBasic розглянув k (скільки сусідів враховувати), min_k (мінімум, який треба для прогнозу), метрику схожості (cosine або pearson) і user_based (дивитись на схожих користувачів чи на схожі фільми). Сітку я залишив ту, що була в прикладі.

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [24]:
# TODO 5.1 — Підберіть гіперпараметри SVD.
# Підказка: GridSearchCV(SVD, param_grid_svd,
#                        measures=['rmse', 'mae'], cv=3, n_jobs=-1)
# Після створення викличте .fit(train_data).

# Ваш код:
grid_search_svd = GridSearchCV(SVD, param_grid_svd,
                               measures=['rmse', 'mae'], cv=3, n_jobs=-1)
grid_search_svd.fit(train_data)

In [25]:
# TODO 5.2 — Підберіть гіперпараметри KNNBasic.
# Підказка: використайте param_grid_knn та ті самі метрики/cv.

# Ваш код:
grid_search_knn = GridSearchCV(KNNBasic, param_grid_knn,
                               measures=['rmse', 'mae'], cv=3, n_jobs=-1)
grid_search_knn.fit(train_data)

In [26]:
# TODO 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
# Підказки:
# grid_search_svd.best_score['rmse']
# grid_search_svd.best_params['rmse']
# grid_search_knn.best_score['rmse']

# Ваш код:
print('SVD')
print('  Найкращий CV RMSE:', round(grid_search_svd.best_score['rmse'], 4))
print('  Найкращий CV MAE:', round(grid_search_svd.best_score['mae'], 4))
print('  Найкращі параметри:', grid_search_svd.best_params['rmse'])

print('KNNBasic')
print('  Найкращий CV RMSE:', round(grid_search_knn.best_score['rmse'], 4))
print('  Найкращий CV MAE:', round(grid_search_knn.best_score['mae'], 4))
print('  Найкращі параметри:', grid_search_knn.best_params['rmse'])

SVD
  Найкращий CV RMSE: 0.9434
  Найкращий CV MAE: 0.7442
  Найкращі параметри: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}
KNNBasic
  Найкращий CV RMSE: 1.0278
  Найкращий CV MAE: 0.8151
  Найкращі параметри: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | ... | ... | ... |
| KNNBasic | ... | ... | ... |

_Порівняйте якість моделей на перехресній перевірці. Зазначте, які параметри стали найкращими._

Порівняв дві моделі на кросвалідації (cv=3) і побачив, що SVD працює краще: RMSE 0.9434 проти 1.0278 у KNNBasic, а MAE 0.7442 проти 0.8151. Для SVD найкраще підійшли 50 факторів, lr_all=0.01 і reg_all=0.05. Для KNNBasic найкраще вийшли 40 сусідів, мінімум 1 сусід, косинусна схожість і user_based=True. Усі числові параметри опинились на краю сітки.

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [27]:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
# TODO 6.2 — Створіть trainset із train_data.
# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.

# Ваш код:
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']

trainset = train_data.build_full_trainset()

best_svd = SVD(n_factors=best_svd_params['n_factors'],
               lr_all=best_svd_params['lr_all'],
               reg_all=best_svd_params['reg_all'],
               random_state=42)
best_knn = KNNBasic(k=best_knn_params['k'],
                    min_k=best_knn_params['min_k'],
                    sim_options=best_knn_params['sim_options'])

best_svd.fit(trainset)
best_knn.fit(trainset)
print('Моделі навчено на повній навчальній вибірці')

Computing the cosine similarity matrix...
Done computing similarity matrix.
Моделі навчено на повній навчальній вибірці


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [28]:
# TODO 7.1 — Отримайте svd_predictions та knn_predictions.
# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
# Підказка: accuracy.rmse(predictions, verbose=False)
#           accuracy.mae(predictions, verbose=False)

# Ваш код:
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)
knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

In [29]:
# TODO 7.3 — Сформуйте DataFrame results із колонками:
# Model, RMSE, MAE.
# Відсортуйте за RMSE та покажіть порівняння.

# Ваш код:
results = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic'],
    'RMSE': [svd_rmse, knn_rmse],
    'MAE': [svd_mae, knn_mae]
}).sort_values('RMSE').reset_index(drop=True)
display(results.round(4))

,Model,RMSE,MAE
0,SVD,0.9192,0.7234
1,KNNBasic,1.0120,0.7984


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | ... | ... |
| KNNBasic | ... | ... |

**Порівняння з CV:** _Прокоментуйте, чи суттєво відрізняються оцінки на кросвалідації та незалежній тестовій вибірці._

Перевірив обидві моделі на тесті, і там SVD теж кращий: RMSE 0.9192 проти 1.0120. На тесті результати трохи кращі, ніж на CV (у SVD було 0.9434, а стало 0.9192), але різниця незначна.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [30]:
# Початковий ID можна змінити
user_id = '196'

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
# TODO 8.2 — Перевірте, що користувач присутній у trainset.
# TODO 8.3 — Знайдіть неоцінені ним фільми.
# TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
# Бажані колонки результату: item_id, predicted_rating.
# Передбачте ситуацію, коли обраного user_id немає у trainset.

# Ваш код:
user_id = '196'

if grid_search_svd.best_score['rmse'] <= grid_search_knn.best_score['rmse']:
    rec_model, rec_name = best_svd, 'SVD'
else:
    rec_model, rec_name = best_knn, 'KNNBasic'
print('Модель для рекомендацій:', rec_name)

try:
    inner_uid = trainset.to_inner_uid(user_id)
except ValueError:
    inner_uid = None
    print(f'Користувача {user_id} немає у навчальній вибірці, рекомендації неможливі.')

if inner_uid is not None:
    rated_items = {trainset.to_raw_iid(iid) for iid, _ in trainset.ur[inner_uid]}
    all_items = [trainset.to_raw_iid(iid) for iid in trainset.all_items()]
    unrated_items = [iid for iid in all_items if iid not in rated_items]
    print('Оцінено фільмів:', len(rated_items), '| неоцінених:', len(unrated_items))

    # 8.4 прогноз та топ-10
    predicted = [(iid, rec_model.predict(uid=user_id, iid=iid).est) for iid in unrated_items]
    top10 = (pd.DataFrame(predicted, columns=['item_id', 'predicted_rating'])
             .sort_values('predicted_rating', ascending=False)
             .head(10).reset_index(drop=True))
    display(top10.round(3))

Модель для рекомендацій: SVD
Оцінено фільмів: 29 | неоцінених: 1624


,item_id,predicted_rating
0,320,4.490
1,64,4.471
2,480,4.414
3,313,4.402
4,513,4.400
5,483,4.398
6,318,4.392
7,656,4.385
8,22,4.377
9,272,4.376


## 9. Перевірка виконання

Позначте після реалізації:

- [ ] Дані `ml-100k` завантажено та перевірено.
- [ ] Є розбиття 80/20 з фіксованим `random_state`.
- [ ] `Reader`, `train_data` і `testset` створені коректно.
- [ ] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [ ] Для SVD і KNNBasic показано параметри та результати CV.
- [ ] Моделі навчено на повній навчальній частині.
- [ ] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [ ] Наведено порівняльну таблицю та інтерпретацію.
- [ ] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [ ] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:** _Напишіть 5–8 речень з опорою на отримані числові результати._

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

За кросвалідацією я вибрав SVD, бо в нього менша помилка: CV RMSE 0.9434, а в KNNBasic 1.0278. Найкращі параметри для SVD: n_factors=50, lr_all=0.01, reg_all=0.05, а для KNNBasic: k=40, min_k=1, косинусна схожість, user_based=True. На тесті SVD отримав RMSE 0.9192 і MAE 0.7234, а KNNBasic RMSE 1.0120 і MAE 0.7984, тобто SVD кращий і тут. Результати на CV і на тесті близькі, тому можна сказати, що оцінка вийшла стабільною. Для користувача 196 я зробив топ-10 фільмів з 1624, які він ще не оцінював, а прогнозовані оцінки вийшли від 4.376 до 4.490. Є й обмеження: рекомендації будуються тільки на оцінках, тож для нових користувачів і фільмів, яких нема в навчальній вибірці, нормально порекомендувати нічого не вийде.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**